In [ ]:
# Import all necessary libraries
import os
import boto3
import pandas as pd
import json
import time
import numpy as np

In [ ]:
# Configuration
PREDICTIONS_FILE = "readmission_predictions_final.csv"
DEMO_FILE = "demo.csv"
OUTPUT_FILE = "readmission_predictions_with_judge_scores.csv"

# Three judge models — median of their scores is used as the final score
JUDGE_MODELS = {
    "nova_pro":       "us.amazon.nova-pro-v1:0",
    "claude_sonnet":  "us.anthropic.claude-3-5-sonnet-20240620-v1:0",
    "claude_opus":    "us.anthropic.claude-opus-4-5-20251101-v1:0",
}

try:
    client = boto3.client("bedrock-runtime", region_name="us-east-1")
except Exception as e:
    print(f"Error initializing Bedrock client: {e}")
    exit()

In [ ]:
# Load Data
try:
    demo_df = pd.read_csv(DEMO_FILE)
except FileNotFoundError:
    print(f"Error: Required file '{DEMO_FILE}' not found.")
    exit()

try:
    predictions_df = pd.read_csv(PREDICTIONS_FILE)
except FileNotFoundError:
    print(f"Error: Predictions file '{PREDICTIONS_FILE}' not found. Run the prediction script first.")
    exit()

In [ ]:
# Prompt Builder
def get_judge_prompt(row, discharge_note_2: str) -> str:
    """
    Constructs the scoring prompt for the judge LLM using the scoring rubric and called for correct predictions (TP or TN).
    """

    actual = row["actual_readmission"]
    pred   = row["pred_readmission"]

    if actual == "Yes" and pred == "Yes":
        classification_type = "TRUE POSITIVE (TP)"
    else:  # actual == "No" and pred == "No"
        classification_type = "TRUE NEGATIVE (TN)"

    ground_truth_section = (
        f"Actual 2nd Admission Note (Ground Truth Context):\n{discharge_note_2}"
        if discharge_note_2
        else "Actual 2nd Admission Note: Not applicable (True Negative)."
    )

    prompt = f"""
        You are a Clinical Rationale Evaluation Expert. Your task is to score the quality of a
        Readmission Risk Rationale produced by an LLM prediction model.

        NOTE: You are ONLY evaluating rationales where the model's prediction was CORRECT
        ({classification_type}). Focus entirely on the quality of the reasoning.

        ─────────────────────────────────────────
        INPUTS
        ─────────────────────────────────────────
        1. LLM Rationale to Evaluate:
        {row['rationale']}

        2. LLM Numerical Risk Prediction: {row['readmission_risk_percent']}%
        3. Actual Readmission Outcome: {actual}
        4. Prediction Classification: {classification_type}
        5. {ground_truth_section}

        ─────────────────────────────────────────
        SCORING RUBRIC (score each criterion independently)
        ─────────────────────────────────────────
        1. Clinical Accuracy (0–5):
        1 = medically incorrect, implausible, or contradicts basic facts
        3 = mostly correct but missing nuance
        5 = clinically accurate and medically sound

        2. Evidence Grounding (0–5):
        1 = unrelated to provided patient information; hallucinations present
        3 = partially grounded but vague
        5 = strongly grounded in diagnoses, vitals, procedures, or notes

        3. Logical Coherence (0–5):
        1 = disorganized or contradictory
        3 = mostly logical with minor gaps
        5 = clear reasoning, causal relationships well expressed

        4. Completeness & Specificity (0–5):
        1 = generic rationale, missing major factors
        3 = partially complete but misses key contributors
        5 = detailed, specific, and covers relevant clinical risk drivers

        ─────────────────────────────────────────
        STRICT OUTPUT FORMAT — JSON ONLY, no extra text
        ─────────────────────────────────────────
        {{
        "clinical_accuracy":       <integer 1–5>,
        "evidence_grounding":      <integer 1–5>,
        "logical_coherence":       <integer 1–5>,
        "completeness_specificity":<integer 1–5>,
        "total_score":             <sum of the four scores, integer 4–20>,
        "judge_summary":           "One concise sentence explaining the key strength or weakness of the rationale."
        }}
        """
    return prompt

In [ ]:
# Call a single judge model
def call_judge(model_id: str, prompt: str, subject_id, adm_id) -> dict | None:
    """
    Calls one judge model and returns parsed JSON dict.
    """
    try:
        response = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
        )
        raw = response["output"]["message"]["content"][0]["text"]
        # Strip markdown code fences if present
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(cleaned)
    except Exception as e:
        print(f"      ⚠ Judge '{model_id}' failed for {subject_id}_{adm_id}: {e}")
        return None

In [ ]:

# Compute median scores across three judges
SCORE_KEYS = [
    "clinical_accuracy",
    "evidence_grounding",
    "logical_coherence",
    "completeness_specificity",
    "total_score",
]

def compute_median_scores(judge_outputs: list[dict]) -> dict:
    """
    Given a list of parsed judge dicts, returns the median value for each score key.
    Ignores None entries (failed calls).
    """
    valid = [o for o in judge_outputs if o is not None]
    if not valid:
        return {k: None for k in SCORE_KEYS}

    medians = {}
    for key in SCORE_KEYS:
        values = [o[key] for o in valid if key in o and o[key] is not None]
        medians[key] = float(np.median(values)) if values else None
    return medians

In [ ]:

# Main Evaluation Loop
judge_results = []

print(f"─── Starting LLM-as-Judge Evaluation ({len(predictions_df)} records) ───\n")
print("NOTE: Only TRUE POSITIVE and TRUE NEGATIVE predictions are scored.\n")

for index, row in predictions_df.iterrows():
    subject_id = row["subject_id"]
    adm_id     = row["adm_id"]
    actual     = row["actual_readmission"]
    pred       = row["pred_readmission"]

    result_base = {
        "subject_id":              subject_id,
        "clinical_accuracy":       None,
        "evidence_grounding":      None,
        "logical_coherence":       None,
        "completeness_specificity":None,
        "total_score":             None,
        "judge_summary":           None,
        "scored":                  False,
        "skip_reason":             None,
    }

    # Only score when prediction matches ground truth
    if actual != pred:
        classification = (
            "FALSE POSITIVE (FP)" if pred == "Yes" else "FALSE NEGATIVE (FN)"
        )
        print(f"[{index+1}/{len(predictions_df)}] SKIP  {subject_id}_{adm_id} — {classification}, not scored per methodology.")
        result_base["skip_reason"] = classification
        judge_results.append(result_base)
        continue

    # Skip if rationale is missing 
    if pd.isna(row.get("rationale")):
        print(f"[{index+1}/{len(predictions_df)}] SKIP  {subject_id}_{adm_id} — rationale missing.")
        result_base["skip_reason"] = "Missing rationale"
        judge_results.append(result_base)
        continue

    # Retrieve 2nd admission discharge note (for TP cases)
    discharge_note_2 = ""
    if actual == "Yes":
        ds2_row = demo_df[(demo_df.SUB_ID == subject_id) & (demo_df.ADM_ID_1 == adm_id)]
        if not ds2_row.empty:
            val = ds2_row.iloc[0].get("DISCHARGE_NOTE_2", None)
            discharge_note_2 = val if pd.notna(val) else ""

    # Build prompt
    prompt = get_judge_prompt(row, discharge_note_2)

    classification_type = "TRUE POSITIVE (TP)" if actual == "Yes" else "TRUE NEGATIVE (TN)"
    print(f"[{index+1}/{len(predictions_df)}] SCORE {subject_id}_{adm_id} — {classification_type}")

    # Call all three judge models 
    judge_outputs = []
    for model_name, model_id in JUDGE_MODELS.items():
        print(f"   → Calling {model_name}...", end=" ")
        output = call_judge(model_id, prompt, subject_id, adm_id)
        if output:
            print(f"Score: {output.get('total_score')}/20")
        else:
            print("FAILED")
        judge_outputs.append(output)
        time.sleep(1)  # Avoid rate limiting between model calls

    # Compute median across the three judges 
    medians = compute_median_scores(judge_outputs)

    # Collect one representative summary (first successful judge)
    summary = next(
        (o["judge_summary"] for o in judge_outputs if o and "judge_summary" in o),
        "Summary unavailable."
    )

    print(f" Median Total Score: {medians.get('total_score')}/20\n")

    result_base.update({
        **medians,
        "judge_summary": summary,
        "scored": True,
    })
    judge_results.append(result_base)

    time.sleep(1)

In [ ]:
# Merge and Save
judge_df = pd.DataFrame(judge_results)

final_df = predictions_df.merge(
    judge_df[[
        "subject_id",
        "clinical_accuracy",
        "evidence_grounding",
        "logical_coherence",
        "completeness_specificity",
        "total_score",
        "judge_summary",
        "scored",
        "skip_reason",
    ]],
    on="subject_id",
    how="left",
)

final_df.to_csv(OUTPUT_FILE, index=False)

# Summary Statistics
scored_df = final_df[final_df["scored"] == True]

print("\n═══════════════════════════════════════════")
print(f"✅ Evaluation Complete → saved to '{OUTPUT_FILE}'")
print(f"   Records scored:  {scored_df.shape[0]}")
print(f"   Records skipped: {final_df[final_df['scored'] == False].shape[0]}")

if not scored_df.empty:
    print("\n── Mean Scores Across Scored Predictions ──")
    for key in SCORE_KEYS:
        mean_val = scored_df[key].mean()
        label = key.replace("_", " ").title()
        if key == "total_score":
            print(f"   {label:<30} {mean_val:.2f} / 20")
        else:
            print(f"   {label:<30} {mean_val:.2f} / 5")

    # Breakdown by prediction label
    print("\n── Score Breakdown by Label ──")
    for label, group in scored_df.groupby("actual_readmission"):
        direction = "Will Be Readmitted" if label == "Yes" else "Will Not Be Readmitted"
        print(f"\n   {direction} (n={len(group)}):")
        for key in SCORE_KEYS:
            max_pts = 20 if key == "total_score" else 5
            print(f"     {key.replace('_',' ').title():<30} {group[key].mean():.2f} / {max_pts}")

print("═══════════════════════════════════════════\n")